# Stage 1 — TF-IDF + Cosine Similarity Baseline

**Project:** Automatic Short Answer Grading using Semantic and Dependency Analysis  
**Repository:** `proposition-level-asag`  
**Current Stage:** Stage 1 — TF-IDF + Cosine Similarity Baseline

---

### Purpose of This Baseline
This notebook provides an interactive demonstration of **System 1: Simple Baseline**, a surface-level lexical similarity benchmark using Term Frequency-Inverse Document Frequency (TF-IDF) and cosine similarity.

In Automated Short Answer Grading (ASAG), this baseline establishes a reference benchmark. It measures what can be achieved through pure surface vocabulary overlap before introducing linguistic parsing, semantic relationships, or proposition-level analysis.

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> The student answers in this demonstration are synthetic examples designed strictly for pipeline verification and qualitative inspection. They must **not** be treated as empirical benchmark data. No human grades or grade thresholds are assigned.

## 1. Imports and Setup

We import `pandas` for tabular formatting and the public baseline API directly from `baseline.tfidf_cosine`. The underlying mathematical and vectorization implementation remains in the modular package and is not duplicated here.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure the repository root is on sys.path so the baseline package is easily imported
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import the existing public API from the baseline package
from baseline.tfidf_cosine import (
    TfidfCosineBaseline,
    compute_tfidf_cosine_similarity,
)

print(f"Loaded project root: {PROJECT_ROOT}")
print("Successfully imported compute_tfidf_cosine_similarity from baseline.tfidf_cosine.")

Loaded project root: /Users/sarang/Dev/College/MIT_WPU/Projects/AIES/proposition-level-asag
Successfully imported compute_tfidf_cosine_similarity from baseline.tfidf_cosine.


## 2. Research Context

### What is TF-IDF?
**Term Frequency-Inverse Document Frequency (TF-IDF)** is a classical numerical statistic from information retrieval:
- **Term Frequency (TF):** Measures how frequently a word appears in a specific answer.
- **Inverse Document Frequency (IDF):** Penalizes words that appear everywhere (common terms) and boosts terms that appear uniquely in specific documents.

In this pairwise baseline, each answer is represented as a sparse vector in vocabulary space, weighted by TF-IDF scores.

### What Does Cosine Similarity Measure?
**Cosine Similarity** calculates the cosine of the angle between two vectors:
$$\text{Cosine Similarity}(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|}$$
- When two answers share identical words in the same proportions, the angle is $0^\circ$ and cosine similarity is **1.0**.
- When two answers share no words in common, their vectors are orthogonal and similarity is **0.0**.
- The score is bounded in $[0.0, 1.0]$.

### Why is This Used as the Simple ASAG Baseline?
1. **Standard Reference Benchmark:** It represents the simplest surface-level approach that requires no linguistic grammar, semantic relations, or external knowledge bases.
2. **Purely Lexical / Surface-Level:** It operates solely on token strings (bag-of-words). It is **not** the proposed ASAG method, nor does it perform syntactic, semantic, or proposition-level reasoning.

## 3. Reference Answer

We evaluate student answers against a standard database question:
> **Question:** *"What is the purpose of a primary key?"*

In [4]:
# Define the reference answer (ground truth)
reference_answer = "A primary key uniquely identifies each record in a table."

print("Reference Answer:")
print(f'"{reference_answer}"')

Reference Answer:
"A primary key uniquely identifies each record in a table."


## 4. Demo Student Answers

We define five synthetic student answers representing distinct response types:
1. **Identical Answer** (exact lexical match)
2. **Near-Verbatim Answer** (high overlap with minor variations)
3. **Paraphrased Answer** (semantically equivalent with synonymous terms)
4. **Partially Overlapping Answer** (shared terms, incomplete concept)
5. **Unrelated Answer** (completely off-topic response)

> **NOTICE: DEMO DATA — NOT RESEARCH DATA**  
> No human grades or grading categories are assigned to these examples.

In [2]:
# Demo student answers (DEMO DATA — NOT RESEARCH DATA)
demo_answers = [
    {
        "id": "demo_1_identical",
        "category": "Identical Answer",
        "description": "Exact match to reference answer with identical wording.",
        "student_answer": "A primary key uniquely identifies each record in a table."
    },
    {
        "id": "demo_2_near_verbatim",
        "category": "Near-Verbatim Answer",
        "description": "High overlap with minor stylistic and lexical additions.",
        "student_answer": "The primary key uniquely identifies each record in the database table."
    },
    {
        "id": "demo_3_paraphrased",
        "category": "Paraphrased Answer",
        "description": "Semantically equivalent answer using synonyms ('every row' vs 'each record').",
        "student_answer": "A primary key uniquely identifies every row in a table."
    },
    {
        "id": "demo_4_partially_overlapping",
        "category": "Partially Overlapping Answer",
        "description": "Shares key terminology ('table', 'key') but omits core uniqueness condition.",
        "student_answer": "A key is used to organize records inside a table."
    },
    {
        "id": "demo_5_unrelated",
        "category": "Unrelated Answer",
        "description": "Completely off-topic response discussing an unrelated biological fact.",
        "student_answer": "Photosynthesis is the process by which green plants use sunlight to synthesize nutrients."
    }
]

print(f"Loaded {len(demo_answers)} demo student answers.")

Loaded 5 demo student answers.


## 5. Calculate Similarities

We compute the continuous cosine similarity score $[0.0, 1.0]$ for each answer pair using `compute_tfidf_cosine_similarity` imported from `baseline.tfidf_cosine`.

In [5]:
# Calculate similarities dynamically using the existing baseline package
results = []
for item in demo_answers:
    similarity_score = compute_tfidf_cosine_similarity(
        reference_answer=reference_answer,
        student_answer=item["student_answer"]
    )
    results.append({
        "Example ID": item["id"],
        "Category": item["category"],
        "Student Answer": item["student_answer"],
        "Cosine Similarity": similarity_score,
        "Description": item["description"]
    })

print("Successfully calculated cosine similarities.")

Successfully calculated cosine similarities.


## 6. Results Table

Below is the summary of calculated similarity scores rendered as a pandas DataFrame.
The scores are dynamically calculated by the baseline pipeline when the notebook is executed.

In [6]:
# Create and display results DataFrame
df_results = pd.DataFrame(results)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.precision", 6)

# Display table
df_results[["Example ID", "Category", "Student Answer", "Cosine Similarity"]]

,Example ID,Category,Student Answer,Cosine Similarity
0,demo_1_identical,Identical Answer,A primary key uniquely identifies each record in a table.,1.000000
1,demo_2_near_verbatim,Near-Verbatim Answer,The primary key uniquely identifies each record in the database table.,0.474493
2,demo_3_paraphrased,Paraphrased Answer,A primary key uniquely identifies every row in a table.,0.716812
3,demo_4_partially_overlapping,Partially Overlapping Answer,A key is used to organize records inside a table.,0.336097
4,demo_5_unrelated,Unrelated Answer,Photosynthesis is the process by which green plants use sunlight to synthesize nutrients.,0.000000


## 7. Individual Examples & Qualitative Observations

Let us inspect each response type individually to observe how the surface lexical baseline behaves.

### Example 1 — Identical Answer
An exact string match to the reference answer.

In [7]:
ex1 = demo_answers[0]
score1 = compute_tfidf_cosine_similarity(reference_answer, ex1["student_answer"])

print(f'Reference:         "{reference_answer}"')
print(f'Student:           "{ex1["student_answer"]}"')
print(f'Cosine Similarity: {score1:.6f}')
print('\nObservation:')
print('Because all terms, positions, and frequencies match perfectly, cosine similarity is exactly 1.0.')

Reference:         "A primary key uniquely identifies each record in a table."
Student:           "A primary key uniquely identifies each record in a table."
Cosine Similarity: 1.000000

Observation:
Because all terms, positions, and frequencies match perfectly, cosine similarity is exactly 1.0.


### Example 2 — Near-Verbatim Answer
Near-verbatim match containing the core answer with minor phrasing additions (*"The"*, *"database"*).

In [8]:
ex2 = demo_answers[1]
score2 = compute_tfidf_cosine_similarity(reference_answer, ex2["student_answer"])

print(f'Reference:         "{reference_answer}"')
print(f'Student:           "{ex2["student_answer"]}"')
print(f'Cosine Similarity: {score2:.6f}')
print('\nObservation:')
print("Introducing non-overlapping words ('the', 'database') and dropping 'a' causes pairwise TF-IDF")
print("to assign high weights to the non-overlapping terms, noticeably reducing similarity.")

Reference:         "A primary key uniquely identifies each record in a table."
Student:           "The primary key uniquely identifies each record in the database table."
Cosine Similarity: 0.474493

Observation:
Introducing non-overlapping words ('the', 'database') and dropping 'a' causes pairwise TF-IDF
to assign high weights to the non-overlapping terms, noticeably reducing similarity.


### Example 3 — Paraphrased Answer
A semantically equivalent answer using synonymous wording (*"every row"* instead of *"each record"*).

In [9]:
ex3 = demo_answers[2]
score3 = compute_tfidf_cosine_similarity(reference_answer, ex3["student_answer"])

print(f'Reference:         "{reference_answer}"')
print(f'Student:           "{ex3["student_answer"]}"')
print(f'Cosine Similarity: {score3:.6f}')
print('\nLexical Limitation Visible:')
print("The student answer expresses the exact same concept as the reference answer.")
print("However, because 'every' != 'each' and 'row' != 'record', the lexical baseline treats them as")
print("completely distinct dimensions, dropping similarity to ~0.7168.")

Reference:         "A primary key uniquely identifies each record in a table."
Student:           "A primary key uniquely identifies every row in a table."
Cosine Similarity: 0.716812

Lexical Limitation Visible:
The student answer expresses the exact same concept as the reference answer.
However, because 'every' != 'each' and 'row' != 'record', the lexical baseline treats them as
completely distinct dimensions, dropping similarity to ~0.7168.


### Example 4 — Partially Overlapping Answer
Contains shared domain terms (*"key"*, *"records"*, *"table"*), but fails to state the essential definition (unique identification).

In [10]:
ex4 = demo_answers[3]
score4 = compute_tfidf_cosine_similarity(reference_answer, ex4["student_answer"])

print(f'Reference:         "{reference_answer}"')
print(f'Student:           "{ex4["student_answer"]}"')
print(f'Cosine Similarity: {score4:.6f}')
print('\nObservation:')
print("Surface overlap of non-essential terms produces a moderate similarity score (~0.3361),")
print("despite the student omitting the defining concept of uniqueness.")

Reference:         "A primary key uniquely identifies each record in a table."
Student:           "A key is used to organize records inside a table."
Cosine Similarity: 0.336097

Observation:
Surface overlap of non-essential terms produces a moderate similarity score (~0.3361),
despite the student omitting the defining concept of uniqueness.


### Example 5 — Unrelated Answer
A completely off-topic response from another domain.

In [11]:
ex5 = demo_answers[4]
score5 = compute_tfidf_cosine_similarity(reference_answer, ex5["student_answer"])

print(f'Reference:         "{reference_answer}"')
print(f'Student:           "{ex5["student_answer"]}"')
print(f'Cosine Similarity: {score5:.6f}')
print('\nObservation:')
print('With zero shared vocabulary, the orthogonal vectors produce a cosine similarity of exactly 0.0.')

Reference:         "A primary key uniquely identifies each record in a table."
Student:           "Photosynthesis is the process by which green plants use sunlight to synthesize nutrients."
Cosine Similarity: 0.000000

Observation:
With zero shared vocabulary, the orthogonal vectors produce a cosine similarity of exactly 0.0.


## 8. Baseline Observations

Qualitative inspection of the baseline behavior across these demo examples reveals clear structural capabilities and limitations:

### What the Lexical Baseline CAN Capture:
- **Surface Lexical Overlap:** Readily identifies identical or shared terminology.
- **Term Importance via TF-IDF:** Weights domain-specific shared terms higher than common shared terms.
- **Vocabulary Disjointness:** Clearly separates responses with topical overlap from completely disjoint responses (yielding 0.0).

### What the Lexical Baseline CANNOT Capture:
- **Synonymy and Paraphrasing:** Cannot recognize that *"every row"* means the same thing as *"each record"*.
- **Syntactic and Dependency Structure:** Words are treated as an unordered bag; sentence structure and grammatical roles (subject, object, modifier) are completely ignored.
- **Negation and Polarity:** Sentences asserting opposite meanings (e.g., *"cannot contain NULL"* vs. *"can contain NULL"*) have nearly identical bag-of-words representations.
- **Proposition-Level Evaluation:** Cannot evaluate whether information present in the student answer that is absent from the reference answer constitutes:
  1. Correct additional information
  2. Irrelevant additional information
  3. Incorrect or misconception-related information

*Note: These observations are qualitative illustrations based on demo examples, not empirical claims of performance on research datasets.*

## 9. Research Significance

In the research methodology outlined in `GEMINI.md`, establishing this TF-IDF baseline serves as the first comparison system.

### Methodological Context
ASAG research requires comparing proposed methods against simpler baselines to determine whether increased complexity yields genuine improvements. The research progression consists of:
1. **System 1 (This Baseline):** Simple surface-level TF-IDF + cosine similarity.
2. **System 2 (Base ASAG):** Classical semantic and dependency-based pipeline (inspired by Basak et al., 2019) incorporating preprocessing, WordNet semantic relationships, dependency parsing, triple matching, and rule-based scoring.
3. **System 3 (Proposed ASAG):** Proposition-level extension designed to evaluate unmatched student information (correct additional, irrelevant, or incorrect).

Establishing this baseline before implementing the Base ASAG system ensures a transparent and reproducible reference point. Whether later systems outperform this baseline is an experimental question to be determined during benchmark evaluation against human grading.

## 10. Conclusion

Stage 1 implementation is complete:
- The TF-IDF + cosine similarity baseline is implemented in `baseline/tfidf_cosine/` and verified with unit tests.
- This interactive notebook demonstrates baseline behavior across five characteristic answer types.
- The pipeline outputs a continuous similarity metric $[0.0, 1.0]$ without inventing arbitrary grade thresholds or human marks.

### Distinction:
- **Observed Demo Behavior:** The baseline separates exact lexical matches from unrelated answers, but penalizes valid paraphrasing and is blind to sentence structure.
- **Future Research Hypothesis:** The upcoming Base ASAG and Proposed ASAG systems are hypothesized to address these limitations, which will be tested in subsequent experimental stages.